# **1. Modelos de previsão**

Este notebook prevê o público do próximo jogo em casa de cada clube da Série A e compara previsões feitas só com a série de público com dois modelos que acrescentam o contexto do jogo. Cada previsão do modelo final é um número de torcedores e um intervalo de previsão de 80%.

| modelo | ideia | entradas |
|---|---|---|
| último jogo | o próximo público é igual ao último público regular em casa | só o histórico |
| média móvel | o próximo público é a média dos últimos cinco públicos regulares em casa | só o histórico |
| suavização exponencial | média ponderada de todos os públicos anteriores, peso 0,7 a cada passo para trás | só o histórico |
| mediana do clube | mediana do clube nas temporadas de treino, sem dinâmica | só o nível do clube |
| regressão com defasagens | regressão ridge do log do público sobre as defasagens e todas as variáveis exógenas | todos os grupos, linear |
| gradient boosting | LightGBM sobre o log do público ou sobre sua razão em relação à média móvel | todos os grupos, não linear |

A avaliação usa janela expansiva por temporada. Cada temporada de 2024 a 2026 é prevista por modelos treinados apenas com as temporadas anteriores: 2024 serve para escolher o alvo e os hiperparâmetros, e 2025 e 2026 são as temporadas de teste reportadas ao final.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

### **1.1. Configuração**

O intervalo busca 80% de cobertura (do 10º ao 90º percentil) e é calibrado por regressão quantílica conformal na temporada anterior à de teste, de modo que nenhum dado de teste define sua largura.

In [2]:
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"
OUTPUT = PROCESSED / "forecasting"
OUTPUT.mkdir(parents=True, exist_ok=True)

TUNE_SEASON = 2024
TEST_SEASONS = [2025, 2026]
FIRST_SEASON = 2022
ALPHA = 0.2
SEED = 7

features = pd.read_csv(PROCESSED / "features.csv", parse_dates=["date", "kickoff_local"])
FEATURE_GROUPS = json.loads((PROCESSED / "feature_groups.json").read_text())
ALL_FEATURES = [c for group in FEATURE_GROUPS.values() for c in group]
data = features[features["crowd_status"].eq("regular")].copy().sort_values("kickoff_local").reset_index(drop=True)
data["log_att"] = np.log(data["attendance"])
print(f"jogos regulares em casa: {len(data):,} | clubes: {data['home'].nunique()}")
display(data.groupby("season").agg(jogos=("game_id", "size"), clubes=("home", "nunique"), mediana=("attendance", "median")))

jogos regulares em casa: 1,675 | clubes: 30


,jogos,clubes,mediana
season,,,
2022,375,20,18254.0
2023,367,20,27718.0
2024,376,20,25888.0
2025,380,20,24906.0
2026,177,20,20267.0


## **2. Funções comuns**

Duas escolhas valem para todos os modelos. O **nível base** de um jogo é a média móvel de cinco jogos; quando o clube não tem nenhum público regular na Série A desde 2022 (um promovido no primeiro jogo), usa-se a mediana do primeiro jogo dos promovidos nas temporadas de treino. A função **fold** retorna as linhas de treino e de avaliação de uma temporada.

In [3]:
def fold(season: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    train = data[data["season"].between(FIRST_SEASON, season - 1)]
    test = data[data["season"].eq(season)]
    return train, test


def base_level(train: pd.DataFrame, frame: pd.DataFrame, use_history: bool = True) -> np.ndarray:
    newcomer_level = train.loc[train["att_history_n"].eq(0), "attendance"].median()
    if not np.isfinite(newcomer_level):
        newcomer_level = train["attendance"].median()
    if use_history:
        return frame["att_ma5"].fillna(newcomer_level).to_numpy()
    club_level = train.groupby("home_id")["attendance"].median()
    return frame["home_id"].map(club_level).fillna(newcomer_level).to_numpy()


def mae(y, p) -> float:
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(p))))


def mape(y, p) -> float:
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(p)) / np.asarray(y)))

## **3. Previsões só com a série de público**

A referência é a média móvel dos últimos cinco públicos regulares em casa, a regra que um departamento de ingressos usaria sem modelo. Outras três previsões usam apenas a série de público: o último público, o público suavizado exponencialmente e a mediana do clube nas temporadas de treino, que ignora qualquer dinâmica e só conhece o tamanho do clube. Todas usam o nível dos promovidos quando o clube não tem histórico.

In [4]:
def predict_moving_average(train, test):
    return base_level(train, test)


def predict_last(train, test):
    return test["att_lag1"].fillna(pd.Series(base_level(train, test), index=test.index)).to_numpy()


def predict_smoothing(train, test):
    return test["att_ewm"].fillna(pd.Series(base_level(train, test), index=test.index)).to_numpy()


def predict_club_median(train, test):
    return base_level(train, test, use_history=False)


ma_check = {season: mae(fold(season)[1]["attendance"], predict_moving_average(*fold(season))) for season in [TUNE_SEASON] + TEST_SEASONS}
display(pd.Series(ma_check, name="MAE").round(0).to_frame())

,MAE
2024,7561.0
2025,7648.0
2026,7408.0


## **4. Regressão com defasagens**

A regressão é ajustada sobre o log do público. As entradas são o log das defasagens e das médias móveis do público, o dia da semana e todas as variáveis numéricas dos demais grupos. Defasagens ausentes (os primeiros jogos de 2022 e de um promovido) são preenchidas com o nível base, para que as defasagens sempre carreguem o tamanho do clube; não há efeito fixo por clube, que contaria esse tamanho duas vezes. As demais variáveis são padronizadas depois de uma imputação pela mediana que mantém um indicador de ausência, e a penalidade da ridge é escolhida por validação cruzada interna nas temporadas de treino.

In [5]:
CROWD_LAGS = ["att_lag1", "att_lag2", "att_lag3", "att_ma5", "att_ewm", "att_season_mean", "att_prev_season_mean"]
LOG_COLUMNS = CROWD_LAGS + ["venue_capacity_ratio", "venue_ratio"]
CATEGORICAL = ["weekday"]


def linear_design(frame: pd.DataFrame, columns: list[str], level: np.ndarray) -> pd.DataFrame:
    design = frame[columns].copy()
    for column in set(CROWD_LAGS).intersection(columns):
        design[column] = design[column].fillna(pd.Series(level, index=design.index))
    for column in set(LOG_COLUMNS).intersection(columns):
        design[column] = np.log(design[column])
    return design


def fit_lag_regression(train, columns):
    numeric = [c for c in columns if c not in CATEGORICAL]
    categorical = [c for c in CATEGORICAL if c in columns]
    transformer = ColumnTransformer([
        ("numeric", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), numeric),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical),
    ])
    model = make_pipeline(transformer, RidgeCV(alphas=np.logspace(-2, 3, 30)))
    model.fit(linear_design(train, columns, base_level(train, train)), train["log_att"])
    return model


def predict_lag_regression(train, test, columns=None):
    columns = columns or ALL_FEATURES
    model = fit_lag_regression(train, columns)
    return np.exp(model.predict(linear_design(test, columns, base_level(train, test))))


lag_check = {season: mae(fold(season)[1]["attendance"], predict_lag_regression(*fold(season))) for season in [TUNE_SEASON] + TEST_SEASONS}
display(pd.Series(lag_check, name="MAE").round(0).to_frame())

,MAE
2024,6769.0
2025,6847.0
2026,5881.0


## **5. Gradient boosting**

O LightGBM recebe todas as variáveis, mas não a identidade do clube. Quatro clubes sobem a cada temporada e nunca aparecem nos dados de treino, e uma variável de clube os manda para um ramo genérico das árvores (uma primeira versão previa cerca de 18.000 torcedores para o Mirassol, cuja média móvel era de 5.000); as variáveis de histórico já informam ao modelo o tamanho de cada clube. Dois alvos são comparados na temporada de ajuste: o log do público e o log da razão entre o público e o nível base. A razão permite que as árvores aprendam desvios comuns a todos os clubes (uma quarta-feira à noite com chuva reduz qualquer público numa proporção parecida) em vez do tamanho de cada clube.

In [6]:
GBM_PARAMS = {
    "objective": "regression_l1", "learning_rate": 0.03, "num_leaves": 15, "min_child_samples": 20,
    "subsample": 0.8, "subsample_freq": 1, "colsample_bytree": 0.8, "reg_lambda": 1.0,
    "n_estimators": 600, "random_state": SEED, "verbose": -1, "deterministic": True, "force_col_wise": True,
}


def gbm_matrix(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    return frame[columns].astype(float)


def fit_gbm(train, columns, target="ratio", params=None, use_history=True):
    params = {**GBM_PARAMS, **(params or {})}
    base = base_level(train, train, use_history)
    y = np.log(train["attendance"].to_numpy() / base) if target == "ratio" else train["log_att"].to_numpy()
    model = lgb.LGBMRegressor(**params)
    model.fit(gbm_matrix(train, columns), y)
    return model


def predict_gbm(train, test, columns=None, target="ratio", params=None, use_history=True, model=None):
    columns = ALL_FEATURES if columns is None else columns
    model = model or fit_gbm(train, columns, target, params, use_history)
    raw = model.predict(gbm_matrix(test, columns))
    return np.exp(raw) * base_level(train, test, use_history) if target == "ratio" else np.exp(raw)


train, valid = fold(TUNE_SEASON)
tuning = []
for target in ("ratio", "level"):
    for num_leaves in (7, 15, 31):
        for n_estimators in (300, 600, 1200):
            params = {"num_leaves": num_leaves, "n_estimators": n_estimators}
            tuning.append({"target": target, **params, "mae": mae(valid["attendance"], predict_gbm(train, valid, target=target, params=params))})
tuning = pd.DataFrame(tuning).sort_values("mae")
display(tuning.head(8).round(0))
best = tuning.iloc[0]
TARGET = best["target"]
GBM_PARAMS.update({"num_leaves": int(best["num_leaves"]), "n_estimators": int(best["n_estimators"])})
print(f"alvo escolhido: {TARGET} | num_leaves={GBM_PARAMS['num_leaves']} | n_estimators={GBM_PARAMS['n_estimators']}")

,target,num_leaves,n_estimators,mae
14,level,15,1200,5908.0
11,level,7,1200,5939.0
13,level,15,600,5952.0
17,level,31,1200,5968.0
12,level,15,300,5981.0
10,level,7,600,5981.0
16,level,31,600,5994.0
15,level,31,300,6030.0


alvo escolhido: level | num_leaves=15 | n_estimators=1200


## **6. Intervalos de previsão**

Dois modelos quantílicos (10º e 90º percentis) são ajustados com as mesmas variáveis e o mesmo alvo do modelo pontual. A regressão quantílica conformal alarga ou estreita os dois limites por um único valor na escala log, o que faz 80% dos jogos da temporada de calibração caírem dentro do intervalo. A temporada de calibração é a anterior à de teste, prevista por modelos treinados com as temporadas antes dela.

In [7]:
def quantile_bounds(train, test):
    lower = predict_gbm(train, test, target=TARGET, params={"objective": "quantile", "alpha": ALPHA / 2})
    upper = predict_gbm(train, test, target=TARGET, params={"objective": "quantile", "alpha": 1 - ALPHA / 2})
    return np.minimum(lower, upper), np.maximum(lower, upper)


def conformal_interval(season: int):
    calibration_train, calibration = fold(season - 1)
    low, high = quantile_bounds(calibration_train, calibration)
    y = np.log(calibration["attendance"].to_numpy())
    scores = np.maximum(np.log(low) - y, y - np.log(high))
    n = len(scores)
    q = np.quantile(scores, min(1, np.ceil((n + 1) * (1 - ALPHA)) / n))
    train, test = fold(season)
    low, high = quantile_bounds(train, test)
    return low * np.exp(-q), high * np.exp(q), q

## **7. Previsões fora da amostra**

Cada modelo prevê cada temporada de teste depois de treinado com todas as temporadas anteriores. As previsões e os intervalos são guardados jogo a jogo.

In [8]:
rows = []
for season in TEST_SEASONS:
    train, test = fold(season)
    low, high, q = conformal_interval(season)
    frame = test[["game_id", "season", "round", "date", "home_id", "home", "away", "stadium", "attendance", "att_history_n", "venue_usual"]].copy()
    frame["pred_last"] = predict_last(train, test)
    frame["pred_ma"] = predict_moving_average(train, test)
    frame["pred_ewm"] = predict_smoothing(train, test)
    frame["pred_club"] = predict_club_median(train, test)
    frame["pred_lag"] = predict_lag_regression(train, test)
    frame["pred_gbm"] = predict_gbm(train, test, target=TARGET)
    frame["low_gbm"], frame["high_gbm"], frame["conformal_q"] = low, high, q
    rows.append(frame)
predictions = pd.concat(rows, ignore_index=True)
predictions["inside"] = predictions["attendance"].between(predictions["low_gbm"], predictions["high_gbm"])
print(f"jogos de teste: {len(predictions):,}")

jogos de teste: 557


## **8. Resultados**

### **8.1. Erro geral**

Erro absoluto médio em torcedores e erro percentual absoluto médio, por temporada de teste e no conjunto. O intervalo é avaliado pela cobertura observada e pela largura mediana.

In [9]:
MODELS = {
    "pred_last": "Último jogo", "pred_ma": "Média móvel", "pred_ewm": "Suavização exponencial", "pred_club": "Mediana do clube",
    "pred_lag": "Regressão com defasagens", "pred_gbm": "Gradient boosting",
}
summary = []
for season, frame in list(predictions.groupby("season")) + [("2025–26", predictions)]:
    for column, label in MODELS.items():
        summary.append({"season": season, "model": label, "MAE": mae(frame["attendance"], frame[column]), "MAPE": mape(frame["attendance"], frame[column])})
summary = pd.DataFrame(summary)
display(summary.pivot(index="model", columns="season", values="MAE").loc[list(MODELS.values())].round(0))
display(summary.pivot(index="model", columns="season", values="MAPE").loc[list(MODELS.values())].mul(100).round(1))
interval = predictions.groupby("season").apply(lambda f: pd.Series({
    "coverage": f["inside"].mean(), "median_width": (f["high_gbm"] - f["low_gbm"]).median(), "conformal_q": f["conformal_q"].iat[0],
}), include_groups=False)
display(interval.round(3))

season,2025,2026,2025–26
model,,,
Último jogo,8719.0,8229.0,8564.0
Média móvel,7648.0,7408.0,7572.0
Suavização exponencial,7419.0,6951.0,7270.0
Mediana do clube,8129.0,6873.0,7730.0
Regressão com defasagens,6847.0,5881.0,6540.0
Gradient boosting,6643.0,5617.0,6317.0


season,2025,2026,2025–26
model,,,
Último jogo,43.3,44.5,43.7
Média móvel,40.2,40.7,40.4
Suavização exponencial,39.2,38.8,39.1
Mediana do clube,51.8,41.9,48.6
Regressão com defasagens,33.5,26.2,31.2
Gradient boosting,32.4,26.1,30.4


,coverage,median_width,conformal_q
season,,,
2025,0.774,22916.841,0.177
2026,0.887,25385.535,0.195


### **8.2. Erro por clube**

A mesma comparação clube a clube, nas duas temporadas de teste. Um clube entra na tabela com ao menos oito jogos de teste.

In [10]:
by_club = predictions.groupby("home").apply(lambda f: pd.Series({
    "matches": len(f), "median_crowd": f["attendance"].median(),
    **{label: mae(f["attendance"], f[column]) for column, label in MODELS.items()},
    "coverage": f["inside"].mean(),
}), include_groups=False)
by_club = by_club[by_club["matches"].ge(8)]
by_club["gain_vs_ma"] = 1 - by_club["Gradient boosting"] / by_club["Média móvel"]
by_club = by_club.sort_values("gain_vs_ma", ascending=False)
display(by_club.round(3).round({label: 0 for label in MODELS.values()} | {"median_crowd": 0}))
print(f"o gradient boosting supera a média móvel em {(by_club['gain_vs_ma'] > 0).sum()} de {len(by_club)} clubes")

,matches,median_crowd,Último jogo,Média móvel,Suavização exponencial,Mediana do clube,Regressão com defasagens,Gradient boosting,coverage,gain_vs_ma
home,,,,,,,,,,
Vasco da Gama,28.0,19523.0,7585.0,7174.0,6547.0,5273.0,3491.0,4220.0,1.000,0.412
Mirassol,30.0,5041.0,3902.0,2906.0,2941.0,9091.0,2269.0,1944.0,0.633,0.331
Sport,19.0,15434.0,9018.0,7233.0,7122.0,6769.0,5727.0,5054.0,0.684,0.301
Chapecoense,11.0,10578.0,4615.0,4683.0,4523.0,7090.0,8290.0,3455.0,0.909,0.262
Athletico-PR,9.0,24146.0,7997.0,8171.0,7970.0,4736.0,8355.0,6060.0,0.889,0.258
Cruzeiro,28.0,40624.0,13446.0,12543.0,12548.0,13753.0,10561.0,9578.0,0.750,0.236
Fortaleza,19.0,27024.0,11386.0,12025.0,11165.0,10936.0,9703.0,9304.0,0.842,0.226
Ceará,19.0,30021.0,16200.0,11497.0,11074.0,10299.0,9311.0,8983.0,0.684,0.219
Botafogo,27.0,16332.0,7024.0,8896.0,7717.0,9448.0,6972.0,7000.0,0.593,0.213


o gradient boosting supera a média móvel em 21 de 24 clubes


## **9. Exportação e auditoria**

| arquivo | conteúdo |
|---|---|
| `data/processed/forecasting/predictions.csv` | uma linha por jogo de teste com todas as previsões e o intervalo |
| `data/processed/forecasting/summary.csv` | MAE e MAPE por modelo e temporada |
| `data/processed/forecasting/by_club.csv` | erro por clube e ganho sobre a média móvel |
| `data/processed/forecasting/run_manifest.json` | alvo, hiperparâmetros e desenho da avaliação, lidos pelo notebook 05 |

A auditoria confirma que nenhuma temporada de teste foi usada no treino ou no ajuste e que a cobertura do intervalo está próxima do valor nominal.

In [11]:
predictions.to_csv(OUTPUT / "predictions.csv", index=False)
summary.to_csv(OUTPUT / "summary.csv", index=False)
by_club.to_csv(OUTPUT / "by_club.csv")
manifest = {
    "generated_at": pd.Timestamp.now().isoformat(timespec="seconds"),
    "target": TARGET, "gbm_params": GBM_PARAMS, "tune_season": TUNE_SEASON, "test_seasons": TEST_SEASONS,
    "first_season": FIRST_SEASON, "alpha": ALPHA, "tuning_mae": float(best["mae"]),
}
(OUTPUT / "run_manifest.json").write_text(json.dumps(manifest, indent=2))

for season in TEST_SEASONS:
    train, test = fold(season)
    assert train["season"].max() < season, "Uma linha de treino vem da temporada de teste"
assert TUNE_SEASON < min(TEST_SEASONS), "A temporada de ajuste coincide com as de teste"
assert abs(predictions["inside"].mean() - (1 - ALPHA)) < 0.1, f"Cobertura do intervalo {predictions['inside'].mean():.2f} distante de {1 - ALPHA:.2f}"
assert predictions[list(MODELS)].notna().all().all(), "Há previsão ausente"